In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from PIL import Image
import matplotlib.pyplot as plt 

In [2]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])

MNIST_test = datasets.MNIST(root='data', train=False, download=True, transform=transform)
MNIST_train = datasets.MNIST(root='data', train=True, download=True, transform=transform)

In [17]:
test_loader = torch.utils.data.DataLoader(dataset=MNIST_test, batch_size=1024, num_workers=4)
train_loader = torch.utils.data.DataLoader(dataset=MNIST_train, batch_size=64, num_workers=4)

print(f"Training batches per epoch: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Train batch size: 64, Test batch size: 1024")

Training batches per epoch: 938
Test batches: 10
Train batch size: 64, Test batch size: 1024


In [18]:
class NN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5)
    self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5)
    self.fc1 = nn.Linear(1024, 128)
    self.fc2 = nn.Linear(128, 10)
  
  def forward(self, x):                # input (1, 28, 28)
    x = self.conv1(x)                # batch (32, 24, 24)
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=2) # batch (32, 12, 12)
    x = self.conv2(x)                # batch (64, 8, 8)
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=2) # batch (64, 4, 4)
    x = torch.flatten(x, start_dim=1)  # batch (1024)
    x = self.fc1(x)                    # batch (128)
    x = self.fc2(x)                    # batch (10)
    return F.log_softmax(x, dim=1)


In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NN().to(device)
optimizer = torch.optim.Adadelta(model.parameters())

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Device: cuda
GPU Name: Tesla V100-PCIE-32GB
GPU Memory: 34.07 GB
Total model parameters: 184,586
Trainable parameters: 184,586


In [20]:
def train(epoch=None):
  model.train()
  total_loss = 0
  for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = F.nll_loss(output, target)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    
    if batch_idx % 500 == 0 and epoch is not None:
      print(f"Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")
  
  avg_loss = total_loss / len(train_loader)
  if epoch is not None:
    print(f"Epoch {epoch} - Average Loss: {avg_loss:.6f}")

In [21]:
def test():
  model.eval()
  test_loss = 0
  correct = 0
  with torch.no_grad():
    for data, target in test_loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      test_loss += F.nll_loss(output, target, reduction='sum').item()
      pred = output.argmax(dim=1, keepdim=True)
      correct += pred.eq(target.view_as(pred)).sum().item()
      
  test_loss /= len(test_loader.dataset)

  print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
      test_loss, correct, len(test_loader.dataset),
      100. * correct / len(test_loader.dataset)))

In [22]:
import time

def main():
  epochs = 10
  start_time = time.time()
  print(f"Starting training for {epochs} epochs...")
  print("=" * 50)
  
  for epoch in range(1, epochs + 1):
    print(f"Epoch {epoch}/{epochs} starting\n")
    epoch_start = time.time()
    train(epoch)
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch}/{epochs} completed in {epoch_time:.2f}s\n")
  
  test()
  
  total_time = time.time() - start_time
  print("=" * 50)
  print(f"Total training time: {total_time:.2f}s ({total_time/60:.2f} minutes)")

In [23]:
main()

Starting training for 10 epochs...
Epoch 1/10 starting

Epoch 1 [0/60000] Loss: 2.312822
Epoch 1 [32000/60000] Loss: 0.047009
Epoch 1 - Average Loss: 0.130890
Epoch 1/10 completed in 4.54s

Epoch 2/10 starting

Epoch 2 [0/60000] Loss: 0.072974
Epoch 2 [32000/60000] Loss: 0.008527
Epoch 2 - Average Loss: 0.038679
Epoch 2/10 completed in 4.45s

Epoch 3/10 starting

Epoch 3 [0/60000] Loss: 0.011585
Epoch 3 [32000/60000] Loss: 0.005655
Epoch 3 - Average Loss: 0.025745
Epoch 3/10 completed in 4.49s

Epoch 4/10 starting

Epoch 4 [0/60000] Loss: 0.010724
Epoch 4 [32000/60000] Loss: 0.004671
Epoch 4 - Average Loss: 0.019626
Epoch 4/10 completed in 4.45s

Epoch 5/10 starting

Epoch 5 [0/60000] Loss: 0.000910
Epoch 5 [32000/60000] Loss: 0.021148
Epoch 5 - Average Loss: 0.015270
Epoch 5/10 completed in 4.44s

Epoch 6/10 starting

Epoch 6 [0/60000] Loss: 0.007269
Epoch 6 [32000/60000] Loss: 0.007992
Epoch 6 - Average Loss: 0.012128
Epoch 6/10 completed in 4.43s

Epoch 7/10 starting

Epoch 7 [0/600